 # Notebook 09 - PPI scanning dual-frequency LWC retrieval



 Companion to notebook 08 (RHI). Same retrieval, same gridding scheme

 (Lamer, Tatarevic, Jo & Kollias 2014, AMT 7, 1089-1103), different geometry.



 ## What changes relative to the RHI notebook



 1. **Elevation is CONSTANT, azimuth varies.** The ray index is azimuth.

 2. **A PPI ray is NOT at constant height.** At fixed elevation theta,

    z = r sin(theta), so height climbs along the ray. This is what makes the

    retrieval possible at all - a truly horizontal ray would have no vertical

    DWR gradient to differentiate.

 3. **All rays share one height axis.** Because theta is constant, z depends

    only on the gate index. The polar array is therefore ALREADY an

    (azimuth, height) array - no interpolation is needed to get the ensemble

    of vertical profiles. Gridding is only needed for the (x, y) plan view.

 4. **Off-zenith angle is constant** = 90 - theta, so Lamer's 25-30 deg

    confidence limit either passes or fails for the whole scan.

 5. **Cressman radius of influence** uses the AZIMUTHAL beamwidth and the

    horizontal projection of the gate length.



 ## What does NOT change



 - The retrieval (`retrieve_lwc_zhu` on slant range -> dDWR/ds)

 - Lamer's post-processing order (Sect. 2)

 - Cressman weights and the Fg formula (Appendix A3-A4)

 - Along-time gridding logic (Sect. 3.3.2)

 - Lamer Sect. 5: retrieve in NATIVE POLAR coordinates, grid only afterwards



 ## What is NOT implemented



 **No beam-mismatch correction.** Neither this notebook nor notebook 08

 corrects for the Ka/W beamwidth mismatch (0.84 vs 0.55 deg). Step 6 only

 QUANTIFIES it. See the note at the end of Step 6.



 Save as notebooks_<DATASET>/09_ppi_scanning_retrieval.ipynb

In [ ]:
import sys
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.zhu import retrieve_lwc_zhu

Processed_Dataset = "20230930_PPI"
PROC = Path(f"../data/processed/{Processed_Dataset}")

# --- instrument -----------------------------------------------------------
HPBW_KA_DEG = 0.84
HPBW_W_DEG  = 0.55
Z_RADAR_M   = 0.0
EARTH_RADIUS_M = 6_371_000.0      # Lamer Eq. (1)

# --- scan geometry --------------------------------------------------------
# Kollias et al. (2014) Sect. 4d / Lamer Sect. 3.3.1: step one beamwidth per
# integration period. Use the SMALLER beamwidth so neither radar is
# undersampled.
AZI_STEP_DEG = HPBW_W_DEG

# Azimuth convention. Lamer Eqs. (4)-(5) use the MATHEMATICAL convention
# (counterclockwise from east). If the files store METEOROLOGICAL azimuth
# (clockwise from north), set this True and it is converted below.
AZIMUTH_IS_METEOROLOGICAL = True

# --- grid (user-dependent) ------------------------------------------------
DX_M, DY_M = 50.0, 50.0
X_MIN_M, X_MAX_M = -2000.0, 2000.0
Y_MIN_M, Y_MAX_M = -2000.0, 2000.0

# --- retrieval ------------------------------------------------------------
N_DEFAULT       = 9
MIN_CLOUD_GATES = 3
EDGE_PAD        = 1
GRID_METHOD     = "cressman"      # "cressman" (App. A3) or "maximum" (App. A1)



In [ ]:
# ---- scan configuration -------------------------------------------------
SCAN_TYPE = "ppi"          # "zenith" | "ppi" | "rhi"

if SCAN_TYPE == "zenith":
    Dataset_94GHz     = "20230930_94GHz"
    Dataset_35GHz     = "20230930_35GHz"
    Processed_Dataset = "20230930"
    SCAN_TOKEN        = "ZEN"
    ELEV_DEG          = 90.0
    TIME_AVERAGE_S    = 30.0          # rolling mean over profiles is legal
else:
    Dataset_94GHz     = "20230930_94GHz_PPI"
    Dataset_35GHz     = "20230930_35GHz_PPI"
    Processed_Dataset = "20230930_PPI"
    SCAN_TOKEN        = "PPI"         # <-- replace with the answer from 0.1
    ELEV_DEG          = 75.0          # <-- replace with the answer from 0.2
    TIME_AVERAGE_S    = 0.0           # each ray stands alone

SIN_ELEV = np.sin(np.radians(ELEV_DEG))
print(f"SCAN_TYPE={SCAN_TYPE}  elev={ELEV_DEG} deg  sin(elev)={SIN_ELEV:.4f}")

 ## Step 1 - Load the scan data



 Same file structure as notebook 08. Variables on dims (Time, range), where

 `Time` indexes RAYS (azimuths here) and `range` is SLANT range:



   Ze_W, Ze_Ka, DWR, common_mask, alpha_W, alpha_Ka, kappa_W, kappa_Ka,

   T_cloud, SLv_W, SLv_Ka, LDR_Ka



 plus per-ray `elevation` and `azimuth` in degrees.

In [ ]:
suffix = "" if SCAN_TYPE == "zenith" else "_scan"
ds_retrieval_scan = xr.open_dataset(
    PROC / f"retrieval_input_4km_alpha_kappa_calculated{suffix}.nc")

TR      = lambda a: a.transpose("Time", "range")
target  = TR(ds_retrieval_scan["DWR"])
to_grid = lambda a: TR(a).reindex_like(target)

rng_m    = target["range"].values.astype(float)     # SLANT range
elev_deg = ds_retrieval_scan["elevation"].values
azi_deg  = ds_retrieval_scan["azimuth"].values

n_ray, n_gate = target.shape
dr_m = float(np.median(np.diff(rng_m)))

# elevation should be essentially constant for a PPI
ELEV_DEG = float(np.median(elev_deg))
elev_spread = float(elev_deg.max() - elev_deg.min())

print(f"{n_ray} rays x {n_gate} gates")
print(f"slant range  {rng_m[0]:.0f}-{rng_m[-1]:.0f} m, {dr_m:.1f} m gates")
print(f"elevation    {ELEV_DEG:.2f} deg  (spread {elev_spread:.3f} deg)")
print(f"azimuth      {azi_deg.min():.1f}-{azi_deg.max():.1f} deg "
      f"(sector {azi_deg.max()-azi_deg.min():.1f} deg)")
print(f"azi step     {np.median(np.abs(np.diff(azi_deg))):.3f} deg "
      f"(target {AZI_STEP_DEG:.2f})")

if elev_spread > 0.2:
    print("  WARNING: elevation is not constant - is this really a PPI?")

# Lamer Sect. 3.3.1: gaps "seldom wider than two times" the beamwidth
gap = np.abs(np.diff(azi_deg))
gap = np.where(gap > 180, 360 - gap, gap)        # wrap-around safe
print(f"rays with gap > 2x beamwidth: {(gap > 2*AZI_STEP_DEG).sum()} "
      f"of {len(gap)}")



In [ ]:
SWEEP = 0
sel = ds_retrieval_scan["sweep_id"].values == SWEEP
ds_retrieval_scan = ds_retrieval_scan.isel(Time=sel)
print(f"sweep {SWEEP}: {sel.sum()} rays, "
      f"{ds_retrieval_scan['Time'].values[0]} to {ds_retrieval_scan['Time'].values[-1]}")

 ## Step 2 - Polar to Cartesian coordinates



 Lamer Sect. 3.3.1, Eqs. (1)-(5), 4/3 effective Earth radius:



     a_ef = 4/3 a

     z    = (r^2 + a_ef^2 + 2 r a_ef sin(theta))^0.5 - a_ef

     s    = a_ef asin( r cos(theta) / (a_ef + z) )

     x    = s cos(phi),   y = s sin(phi)



 With theta CONSTANT, z and s depend only on the gate index, so both are 1-D.

 Every ray shares the same height axis - the key structural difference from

 the RHI case.

In [ ]:
a_ef  = (4.0 / 3.0) * EARTH_RADIUS_M
th    = np.radians(ELEV_DEG)

z_gate = np.sqrt(rng_m**2 + a_ef**2 + 2.0 * rng_m * a_ef * np.sin(th)) - a_ef
z_gate = z_gate + Z_RADAR_M                                 # (gate,) height
s_gate = a_ef * np.arcsin(rng_m * np.cos(th) / (a_ef + z_gate))  # (gate,) horiz

# azimuth -> mathematical convention for Eqs. (4)-(5)
phi_deg = (90.0 - azi_deg) if AZIMUTH_IS_METEOROLOGICAL else azi_deg
phi     = np.radians(phi_deg)[:, None]                      # (ray, 1)

x_obs = s_gate[None, :] * np.cos(phi)                       # (ray, gate)
y_obs = s_gate[None, :] * np.sin(phi)
z_obs = np.repeat(z_gate[None, :], n_ray, axis=0)           # for convenience

# height resolution of the retrieval window
dz_gate = dr_m * np.sin(th)
print(f"height       {z_gate[0]:.0f}-{z_gate[-1]:.0f} m")
print(f"horizontal   {s_gate[0]:.0f}-{s_gate[-1]:.0f} m")
print(f"vertical gate spacing: {dz_gate:.1f} m "
      f"(slant {dr_m:.1f} m x sin {ELEV_DEG:.0f} deg)")
print(f"a {N_DEFAULT}-gate window spans {N_DEFAULT*dz_gate:.0f} m vertically "
      f"({N_DEFAULT*dr_m:.0f} m along beam)")

# sounding fields must track HEIGHT; with constant elevation that means they
# should depend only on the gate index, identically for every ray
alpha_w_v = to_grid(ds_retrieval_scan["alpha_W"]).values
k = n_gate // 2
spread = np.nanmax(alpha_w_v[:, k]) - np.nanmin(alpha_w_v[:, k])
print(f"alpha_W at gate {k} (z={z_gate[k]:.0f} m): "
      f"spread across rays {spread:.4f} dB/km  -> should be ~0")



 ## Step 3 - Post-processing in polar coordinates



 Lamer Sect. 2, unchanged from notebook 08:

   (i)   feature mask + range-AZIMUTH spatial filter

   (ii)  gaseous attenuation - handled in the LWC equation via Delta_alpha,

         so Ze is NOT corrected here (doing both double-counts the gas)

   (iii) Doppler unfolding - not needed

   (iv)  insect filter via LDR where T > 5 C

In [ ]:
SNR_MARGIN_DB = 3.0

ze_ka = to_grid(ds_retrieval_scan["Ze_Ka"]).values
ze_w  = to_grid(ds_retrieval_scan["Ze_W"]).values
sl_ka = 10.0 * np.log10(to_grid(ds_retrieval_scan["SLv_Ka"]).values)
sl_w  = 10.0 * np.log10(to_grid(ds_retrieval_scan["SLv_W"]).values)

mask_scan = (np.isfinite(ze_ka) & np.isfinite(ze_w)
             & (ze_ka > sl_ka + SNR_MARGIN_DB)
             & (ze_w  > sl_w  + SNR_MARGIN_DB))
n0 = int(mask_scan.sum())

# range-azimuth spatial filter: keep bins with >= 2 detected neighbours
nb = np.zeros_like(mask_scan, dtype=int)
nb[1:, :]  += mask_scan[:-1, :]
nb[:-1, :] += mask_scan[1:, :]
nb[:, 1:]  += mask_scan[:, :-1]
nb[:, :-1] += mask_scan[:, 1:]
mask_scan &= (nb >= 2)
print(f"(i)  feature mask: {n0} -> {int(mask_scan.sum())} bins after filter")

T_C = to_grid(ds_retrieval_scan["T_cloud"]).values - 273.15
LDR_THRESHOLD_DB = -20.0
if "LDR_Ka" in ds_retrieval_scan:
    ldr = to_grid(ds_retrieval_scan["LDR_Ka"]).values
    insect = (T_C > 5.0) & np.isfinite(ldr) & (ldr > LDR_THRESHOLD_DB)
    print(f"(iv) LDR insect filter: removed {int(insect.sum())} bins")
else:
    insect = (T_C > 5.0) & (target.values < -1.0)
    print(f"(iv) no LDR, DWR < -1 dB fallback: removed {int(insect.sum())} bins")
mask_scan &= ~insect

print(f"final cloudy bins: {int(mask_scan.sum())} ({mask_scan.mean():.1%})")
mask_scan_da = xr.DataArray(mask_scan, dims=target.dims, coords=target.coords)



 ## Step 4 - LWC retrieval in native polar coordinates



 Identical to notebook 08. `retrieve_lwc_zhu` differentiates DWR against the

 `range` coordinate, which is SLANT range, giving dDWR/ds along the

 propagation path.



     LWC = (0.5 dDWR/ds - Delta_alpha) / Delta_kappa



 Each ray is one independent retrieval. Because the geometry is identical for

 every ray, this is an ENSEMBLE of statistically identical retrievals - the

 main analytical advantage of PPI over RHI.

In [ ]:
lwc_polar, ddwr_ds, m_eff, n_windows, n_used = retrieve_lwc_zhu(
    dwr        = target,
    kappa_w    = to_grid(ds_retrieval_scan["kappa_W"]),
    kappa_ka   = to_grid(ds_retrieval_scan["kappa_Ka"]),
    alpha_w    = to_grid(ds_retrieval_scan["alpha_W"]),
    alpha_ka   = to_grid(ds_retrieval_scan["alpha_Ka"]),
    cloud_mask = mask_scan_da,
    N_default  = N_DEFAULT,
    min_cloud_gates = MIN_CLOUD_GATES,
    edge_pad   = EDGE_PAD,
)

lwc_v = lwc_polar.where(mask_scan_da).values
print(f"retrieved bins {int(np.isfinite(lwc_v).sum())}")
print(f"LWC median {np.nanmedian(lwc_v):+.3f}, std {np.nanstd(lwc_v):.3f} g/m3")
print(f"N used: median {float(n_used.where(n_used>0).median()):.0f}, "
      f"min {int(n_used.where(n_used>0).min())}, max {int(n_used.max())}")



 ## Step 5a - Ensemble of vertical profiles (no interpolation needed)



 Because elevation is constant, `lwc_v` is already an (azimuth, height)

 array. Relabelling the range axis as height is exact - no gridding, no

 smoothing, no Lamer Sect. 5 concern.



 This is the primary product for the error budget: N_ray independent

 retrievals of the same quantity at the same geometry.

In [ ]:
lwc_ensemble = xr.DataArray(
    lwc_v, dims=["azimuth", "height"],
    coords={"azimuth": azi_deg, "height": z_gate},
    name="LWC", attrs={"units": "g m-3"})

prof_median = np.nanmedian(lwc_v, axis=0)
prof_std    = np.nanstd(lwc_v, axis=0)
prof_n      = np.isfinite(lwc_v).sum(axis=0)

print(f"{'height':>8}{'rays':>7}{'median LWC':>12}{'std':>8}{'sem':>8}")
for j in range(0, n_gate, max(n_gate // 12, 1)):
    if prof_n[j] < 5:
        continue
    sem = prof_std[j] / np.sqrt(prof_n[j])
    print(f"{z_gate[j]:>8.0f}{prof_n[j]:>7}{prof_median[j]:>12.3f}"
          f"{prof_std[j]:>8.3f}{sem:>8.3f}")

print("\nstd across rays = per-ray retrieval noise + real azimuthal variability")
print("sem = uncertainty on the SCAN-MEAN profile, which is the deliverable")



 ## Step 5b - Grid to a Cartesian plan view (x, y)



 Lamer Appendix A, adapted to the horizontal plane. The radial extent of a

 bin projects to dr cos(theta); the cross-beam extent uses the AZIMUTHAL

 beamwidth at horizontal distance s:



     R    = max(Rd1, Rd2)                                             (A5)

     Rd1  = sqrt( (dx^2 + dy^2) / 4 )                                 (A6)

     Rd2  = sqrt( (dr cos th)^2

                  + (s + 0.5 dr cos th)^2 sin^2(0.5 max(dphi, bw)) )  (A6)

     w    = (R^2 - d^2) / (R^2 + d^2)   for d^2 <= R^2, else 0        (A4)

     Fg   = sum(w F) / sum(w)                                         (A3)



 Every grid cell sits at a DIFFERENT height, because z is a function of

 radial distance. `z_grid` below records it.

In [ ]:
grid_x = np.arange(X_MIN_M, X_MAX_M + DX_M, DX_M)
grid_y = np.arange(Y_MIN_M, Y_MAX_M + DY_M, DY_M)

dphi_deg = float(np.median(np.abs(np.diff(azi_deg))))
ang      = np.radians(0.5 * max(dphi_deg, AZI_STEP_DEG))
dr_h     = dr_m * np.cos(th)                      # horizontal gate extent
Rd1      = np.sqrt((DX_M**2 + DY_M**2) / 4.0)
Rd2      = np.sqrt(dr_h**2 + (s_gate + 0.5 * dr_h)**2 * np.sin(ang)**2)
R_infl   = np.maximum(Rd1, Rd2)                   # (gate,)

print(f"Rd1 = {Rd1:.1f} m (grid)")
print(f"Rd2 = {Rd2.min():.1f}-{Rd2.max():.1f} m (radar volume, horizontal)")
print(f"R   = {R_infl.min():.1f}-{R_infl.max():.1f} m")


def grid_polar_xy(field_polar, valid_polar, method=GRID_METHOD):
    """Lamer Appendix A1 (maximum) / A3-A4 (Cressman), horizontal plane."""
    nx, ny = len(grid_x), len(grid_y)
    num = np.zeros((nx, ny))
    den = np.zeros((nx, ny))
    out = np.full((nx, ny), -np.inf)

    for i in range(n_ray):
        for k in range(n_gate):
            if not valid_polar[i, k] or not np.isfinite(field_polar[i, k]):
                continue
            R = R_infl[k]
            xo, yo = x_obs[i, k], y_obs[i, k]
            ix = np.flatnonzero(np.abs(grid_x - xo) <= R)
            iy = np.flatnonzero(np.abs(grid_y - yo) <= R)
            if ix.size == 0 or iy.size == 0:
                continue
            d2 = ((grid_x[ix][:, None] - xo)**2
                  + (grid_y[iy][None, :] - yo)**2)
            inside = d2 <= R**2
            if not inside.any():
                continue
            if method == "cressman":
                w = np.where(inside, (R**2 - d2) / (R**2 + d2), 0.0)     # (A4)
                num[np.ix_(ix, iy)] += w * field_polar[i, k]
                den[np.ix_(ix, iy)] += w
            else:
                blk = out[np.ix_(ix, iy)]
                out[np.ix_(ix, iy)] = np.where(
                    inside, np.maximum(blk, field_polar[i, k]), blk)     # (A1)

    if method == "cressman":
        return np.where(den > 0, num / den, np.nan)                      # (A3)
    return np.where(np.isfinite(out), out, np.nan)


valid_lwc = mask_scan & np.isfinite(lwc_v)

lwc_grid = grid_polar_xy(lwc_v, valid_lwc)
dwr_grid = grid_polar_xy(target.values, mask_scan)
ze_grid  = grid_polar_xy(ze_ka, mask_scan)

# height of each grid cell: from its radial distance, inverting s(r)
XX, YY   = np.meshgrid(grid_x, grid_y, indexing="ij")
s_grid   = np.sqrt(XX**2 + YY**2)
r_grid   = s_grid / np.cos(th)
z_grid   = (np.sqrt(r_grid**2 + a_ef**2
                    + 2.0 * r_grid * a_ef * np.sin(th)) - a_ef + Z_RADAR_M)
z_grid   = np.where(np.isfinite(lwc_grid), z_grid, np.nan)

print(f"grid {len(grid_x)} x {len(grid_y)}, "
      f"{int(np.isfinite(lwc_grid).sum())} filled LWC cells")
print(f"heights sampled by the plan view: "
      f"{np.nanmin(z_grid):.0f}-{np.nanmax(z_grid):.0f} m")



 ## Step 6 - Domain limits and beam geometry



 Lamer Sect. 4.2: for stratus the retrieval standard deviation "is almost

 constant within +/- 1.5 km along the scan range from the radar. Beyond this

 range, the standard deviation grows fast."



 Lamer Sect. 5: robust "up to 25-30 degrees off zenith".



 For a PPI the off-zenith angle is CONSTANT, so that second criterion is a

 single pass/fail for the whole scan.

In [ ]:
off_zenith = 90.0 - ELEV_DEG
print(f"off-zenith angle: {off_zenith:.1f} deg  "
      f"-> {'INSIDE' if off_zenith <= 30 else 'OUTSIDE'} "
      f"Lamer's 25-30 deg confidence range")

print(f"\n{'height':>8}{'slant r':>9}{'horiz s':>9}{'bins':>7}"
      f"{'LWC std':>10}{'Ze med':>9}")
for j in range(0, n_gate, max(n_gate // 12, 1)):
    sel = mask_scan[:, j]
    if sel.sum() < 10:
        continue
    v = lwc_v[sel, j]; v = v[np.isfinite(v)]
    if v.size < 5:
        continue
    print(f"{z_gate[j]:>8.0f}{rng_m[j]:>9.0f}{s_gate[j]:>9.0f}"
          f"{int(sel.sum()):>7}{np.std(v):>10.3f}"
          f"{np.nanmedian(ze_ka[sel, j]):>9.1f}")


In [ ]:
# Beam geometry. QUANTIFIED ONLY - no correction is applied anywhere in this
# notebook. For a PPI these numbers are the same for every ray, which is the
# main reason PPI is easier to characterise than RHI.
D_ka = 2.0 * rng_m * np.tan(np.radians(HPBW_KA_DEG) / 2.0)
D_w  = 2.0 * rng_m * np.tan(np.radians(HPBW_W_DEG)  / 2.0)

print(f"\nbeam diameters and vertical smear at elevation {ELEV_DEG:.0f} deg")
print(f"{'height':>8}{'D_Ka':>8}{'D_W':>8}{'diff':>7}{'ratio':>7}"
      f"{'v-smear Ka':>12}{'v-smear W':>11}")
for j in range(0, n_gate, max(n_gate // 8, 1)):
    vs_ka = D_ka[j] * np.cos(th)
    vs_w  = D_w[j]  * np.cos(th)
    print(f"{z_gate[j]:>8.0f}{D_ka[j]:>8.1f}{D_w[j]:>8.1f}"
          f"{D_ka[j]-D_w[j]:>7.1f}{D_ka[j]/D_w[j]:>7.2f}"
          f"{vs_ka:>12.1f}{vs_w:>11.1f}")

print("\nNOTE: vertical smear = D cos(theta). It is ZERO at zenith and grows")
print("as elevation drops, and it differs between the two radars. It acts")
print("exactly at cloud base and top, where sigma_DWR was measured at")
print("0.426 dB versus 0.223 dB in the cloud interior.")
print("Williams & Vivekanandan (2007) recommend an advection-matched")
print("smoothing kernel to equalise the two beams. NOT implemented here.")



 ## Step 7 - Plot

In [ ]:
fig = plt.figure(figsize=(13, 9))

# plan view, LWC
ax1 = fig.add_subplot(2, 2, 1)
p1 = ax1.pcolormesh(grid_x / 1000, grid_y / 1000, lwc_grid.T,
                    cmap="Blues", shading="auto", vmin=0, vmax=0.6)
plt.colorbar(p1, ax=ax1, label="LWC [g m$^{-3}$]")
for rad in (0.5, 1.0, 1.5, 2.0):
    ax1.add_patch(plt.Circle((0, 0), rad, fill=False, color="k",
                             lw=0.6, ls=":"))
ax1.set_xlabel("x [km]"); ax1.set_ylabel("y [km]")
ax1.set_title(f"PPI plan view, {GRID_METHOD} gridding, "
              f"elev {ELEV_DEG:.0f} deg")
ax1.set_aspect("equal")

# plan view, height of each cell
ax2 = fig.add_subplot(2, 2, 2)
p2 = ax2.pcolormesh(grid_x / 1000, grid_y / 1000, z_grid.T,
                    cmap="magma", shading="auto")
plt.colorbar(p2, ax=ax2, label="height of grid cell [m]")
ax2.set_xlabel("x [km]"); ax2.set_ylabel("y [km]")
ax2.set_title("height sampled (radial distance sets z)")
ax2.set_aspect("equal")

# ensemble: azimuth vs height
ax3 = fig.add_subplot(2, 2, 3)
p3 = ax3.pcolormesh(azi_deg, z_gate, lwc_v.T,
                    cmap="Blues", shading="auto", vmin=0, vmax=0.6)
plt.colorbar(p3, ax=ax3, label="LWC [g m$^{-3}$]")
ax3.set_xlabel("azimuth [deg]"); ax3.set_ylabel("height [m]")
ax3.set_title("ensemble of profiles (no interpolation)")

# scan-mean profile
ax4 = fig.add_subplot(2, 2, 4)
sem = prof_std / np.sqrt(np.maximum(prof_n, 1))
ok = prof_n >= 5
ax4.plot(prof_median[ok], z_gate[ok], "o-", ms=3, label="scan median")
ax4.fill_betweenx(z_gate[ok],
                  prof_median[ok] - sem[ok], prof_median[ok] + sem[ok],
                  alpha=0.3, label="standard error of the mean")
ax4.fill_betweenx(z_gate[ok],
                  prof_median[ok] - prof_std[ok], prof_median[ok] + prof_std[ok],
                  alpha=0.12, label="spread across rays")
ax4.axvline(0, color="k", lw=0.8)
ax4.set_xlabel("LWC [g m$^{-3}$]"); ax4.set_ylabel("height [m]")
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)
ax4.set_title("scan-mean profile")

plt.tight_layout()
plt.show()



 ## Step 8 - Save

In [ ]:
ds_scan_out = xr.Dataset(
    {
        "LWC":          (("x", "y"), lwc_grid),
        "DWR":          (("x", "y"), dwr_grid),
        "Ze_Ka":        (("x", "y"), ze_grid),
        "height_grid":  (("x", "y"), z_grid),
        "LWC_polar":    (("Time", "range"), lwc_v),
        "DWR_polar":    (("Time", "range"), target.values),
        "mask_polar":   (("Time", "range"), mask_scan),
        "LWC_profile_median": (("range",), prof_median),
        "LWC_profile_std":    (("range",), prof_std),
        "LWC_profile_n":      (("range",), prof_n),
        "height":       (("range",), z_gate),
        "horizontal":   (("range",), s_gate),
        "N_used":       (("Time",), n_used.values),
        "azimuth":      (("Time",), azi_deg),
    },
    coords={"x": grid_x, "y": grid_y,
            "Time": target["Time"], "range": rng_m},
    attrs={
        "reference": "Lamer et al. (2014), AMT 7, 1089-1103",
        "scan_type": "PPI, constant elevation",
        "elevation_deg": ELEV_DEG,
        "off_zenith_deg": off_zenith,
        "gridding": f"{GRID_METHOD}, Appendix A, horizontal plane",
        "retrieval_coords": ("native polar, per Lamer Sect. 5: dual-wavelength "
                             "microphysical retrievals should be performed in "
                             "the native coordinate system"),
        "dx_m": DX_M, "dy_m": DY_M,
        "azi_step_deg": dphi_deg,
        "vertical_gate_spacing_m": float(dz_gate),
        "hpbw_ka_deg": HPBW_KA_DEG, "hpbw_w_deg": HPBW_W_DEG,
        "beam_mismatch_corrected": "no",
        "N_default": N_DEFAULT, "edge_pad": EDGE_PAD,
        "snr_margin_db": SNR_MARGIN_DB,
    },
)
out_path = PROC / "ppi_lwc_gridded_scan.nc"
ds_scan_out.to_netcdf(out_path)
print(f"saved -> {out_path}")



 ## Step 9 (later) - Along-time gridding



 Lamer Sect. 3.3.2, identical logic to notebook 08 and simpler here because

 consecutive PPIs repeat exactly the same geometry.



   "The selected temporal resolution dt is preferred to be greater than or

    equal to the average time between two successive scans ... Furthermore,

    dt should not be higher than a quarter of the scan duration."



 Nearest-neighbour along time gives the "measured values", then:

   case 1 - both bracketing values contain detections -> linear interpolation

   case 2 - one contains no detection -> hold each value constant over half

            the interval



 For the drone comparison: with repeated PPIs the drone can hold a fixed

 height and the interpolated field gives LWC at the drone position between

 scans. Compare BOTH the directly measured rays and the interpolated values.

In [ ]:
# SCAN_DURATION_S = ...     # sector width / omega
# DT_S = ...                # >= mean gap between scans, <= SCAN_DURATION_S / 4
